# ***🧮 Utilitários de Precisão Decimal***

In [1]:
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
from math import prod
from typing import List, Union

In [2]:
import unittest

# ***🔍 Funções de análise de precisão***

In [3]:
def contar_algarismos_significativos(numero: Decimal) -> int:
    """
    Retorna o número de algarismos significativos em um número Decimal.
    Zeros à esquerda não são considerados significativos.
    """
    numero_normalizado = numero.normalize()
    digitos = numero_normalizado.as_tuple().digits
    return len(digitos)

def contar_casas_decimais(numero: Decimal) -> int:
    """
    Retorna o número de casas decimais de um número Decimal.
    Se o expoente for negativo, representa casas decimais.
    Se for zero ou positivo, retorna 0.
    """
    expoente = numero.as_tuple().exponent
    return expoente if expoente < 0 else 0

# ***✅ Validação de entrada***

In [4]:
def converter_para_decimal(valores_entrada: List[Union[str, Decimal]]) -> List[Decimal]:
    """
    Valida e converte os elementos da lista para Decimal.
    Lança erro se algum elemento for de tipo inválido ou não puder ser convertido.
    """
    decimais = []
    for valor in valores_entrada:
        if not isinstance(valor, (str, Decimal)):
            raise TypeError(f"Tipo inválido: {type(valor).__name__}. Esperado str ou Decimal.")
        try:
            decimais.append(Decimal(valor))
        except InvalidOperation:
            raise ValueError(f"Não foi possível converter '{valor}' para Decimal.")
    return decimais

# ***➕ Soma com Arredondamento***

In [5]:
def somar_com_arredondamento(valores_entrada: List[Union[str, Decimal]], sinais_aplicados: List[int]) -> Decimal:
    """
    Soma valores decimais com sinais aplicados e arredonda para a menor precisão decimal.
    """
    if len(valores_entrada) != len(sinais_aplicados):
        raise ValueError("Listas 'valores_entrada' e 'sinais_aplicados' devem ter o mesmo tamanho.")

    valores_decimais = converter_para_decimal(valores_entrada)
    sinais_inteiros = [int(s) for s in sinais_aplicados]
    valores_com_sinal = [d * s for d, s in zip(valores_decimais, sinais_inteiros)]

    casas_decimais = [abs(contar_casas_decimais(d)) for d in valores_decimais]
    menor_casas_decimais = min(casas_decimais)
    formato_arredondamento = Decimal(f'1.{"0" * menor_casas_decimais}') if menor_casas_decimais > 0 else Decimal('1')

    soma_final = sum(valores_com_sinal).quantize(formato_arredondamento, rounding=ROUND_HALF_UP)
    return soma_final

# ***✖️ Produto com Arredondamento***

In [6]:
def multiplicar_com_arredondamento(valores_entrada: List[Union[str, Decimal]], expoentes_aplicados: List[int]) -> Decimal:
    """
    Calcula o produto de potências de decimais e arredonda com base na menor significância.
    """
    if len(valores_entrada) != len(expoentes_aplicados):
        raise ValueError("Listas 'valores_entrada' e 'expoentes_aplicados' devem ter o mesmo tamanho.")

    valores_decimais = converter_para_decimal(valores_entrada)
    expoentes_inteiros = [int(e) for e in expoentes_aplicados]

    produto_potencias = prod([d ** e for d, e in zip(valores_decimais, expoentes_inteiros)])

    menor_sigfig_entrada = min(map(contar_algarismos_significativos, valores_decimais))

    # Arredondar o produto para o mesmo número de algarismos significativos da menor entrada
    produto_str = f"{produto_potencias:.{menor_sigfig_entrada}g}"
    produto_final = Decimal(produto_str)

    return produto_final

# ***🧪 Teste***
### O codigo a seguir realiza alguns testes de todas as funções acima

In [7]:
class TestFuncoesDecimais(unittest.TestCase):
    def test_contar_algarismos_significativos(self):
        self.assertEqual(contar_algarismos_significativos(Decimal('123.45')), 5)
        self.assertEqual(contar_algarismos_significativos(Decimal('0.001230')), 3)
        self.assertEqual(contar_algarismos_significativos(Decimal('1000')), 1)

    def test_contar_casas_decimais(self):
        self.assertEqual(contar_casas_decimais(Decimal('123.45')), -2)
        self.assertEqual(contar_casas_decimais(Decimal('1000')), 0)
        self.assertEqual(contar_casas_decimais(Decimal('0.001')), -3)

    def test_converter_para_decimal(self):
        entrada = ['1.23', Decimal('4.56')]
        esperado = [Decimal('1.23'), Decimal('4.56')]
        self.assertEqual(converter_para_decimal(entrada), esperado)

        with self.assertRaises(TypeError):
            converter_para_decimal(['1.23', 4.56])  # float não permitido

    def test_somar_com_arredondamento(self):
        valores = ['1.23', '4.567']
        sinais = [1, 1]
        resultado = somar_com_arredondamento(valores, sinais)
        self.assertEqual(resultado, Decimal('5.80'))

        valores = ['1.2', '1.23']
        sinais = [1, -1]
        resultado = somar_com_arredondamento(valores, sinais)
        self.assertEqual(resultado, Decimal('0.0'))

    def test_multiplicar_com_arredondamento(self):
        valores = ['1.23', '4.5']
        expoentes = [1, 1]
        resultado = multiplicar_com_arredondamento(valores, expoentes)
        self.assertEqual(resultado, Decimal('5.5'))

        valores = ['1.00', '2.0']
        expoentes = [1, 1]
        resultado = multiplicar_com_arredondamento(valores, expoentes)
        self.assertEqual(resultado, Decimal('2.0'))

# ✅ Rodar testes no Jupyter
unittest.main(argv=['first-arg-is-ignored'], exit=False)

.....
----------------------------------------------------------------------
Ran 5 tests in 0.002s

OK


In [8]:
valores = ['12.0', '7.15']
sinais = [1, -1]
resultado = somar_com_arredondamento(valores, sinais)
print (resultado)

4.9


In [9]:
valores = ['12.0001', '700000']
sinais = [1, -1]
resultado = multiplicar_com_arredondamento(valores, sinais)
print (resultado)

0.00002
